# Генератор жизненного цикла клиента для синтетических диалогов и Knowledge Graph

Ноутбук создает синтетический набор клиентов, обращений, диалогов и граф знаний.  
Все данные вымышленные, внешние API не используются.

## 1. Цель работы

Цель работы - создать демонстрационный генератор жизненных циклов клиентов в синтетических банковских диалогах.

В итоговом наборе данных у клиентов разные сценарии поведения: короткие обращения, покупки, жалобы, повторные покупки, уход после негативного опыта и длинные цепочки взаимодействий на несколько месяцев.  
Сгенерированные данные используются для построения Knowledge Graph, интерактивной визуализации клиентского пути и аналитики по продуктам, темам, обращениям и удовлетворенности.

## 2. Установка и импорт библиотек

В ячейке ниже проверяется наличие основных библиотек. Если какой-то пакет отсутствует, он устанавливается автоматически через `pip`.

In [1]:
import sys
import subprocess
import importlib.util

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "faker": "faker",
    "networkx": "networkx",
    "plotly": "plotly",
    "ipywidgets": "ipywidgets",
    "pyvis": "pyvis",
}

missing_packages = [
    pip_name
    for import_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print("Устанавливаются отсутствующие пакеты:", ", ".join(missing_packages))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("Все основные библиотеки уже установлены")

Устанавливаются отсутствующие пакеты: faker, pyvis


In [2]:
import random
import uuid
import html
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from faker import Faker
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from pyvis.network import Network
from IPython.display import display, IFrame, HTML, Markdown

random.seed(42)
np.random.seed(42)
Faker.seed(42)

fake = Faker("ru_RU")
OUTPUT_DIR = Path(".")

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)
pio.renderers.default = "notebook"

print("Импорт завершен")

Импорт завершен


## 3. Справочники и настройки генерации

Ниже задаются сегменты, каналы, продукты, темы, сценарии жизненного цикла и правила для генерации событий.

In [3]:
SEGMENTS = ["new", "regular", "premium", "risk", "sleeping", "problematic", "loyal"]

CHANNELS = [
    "мобильное приложение",
    "сайт",
    "офис",
    "телефон",
    "чат",
    "email",
]

PRODUCTS = [
    "Дебетовая карта",
    "Кредитная карта",
    "Потребительский кредит",
    "Ипотека",
    "Инвестиционный счет",
    "Страхование",
    "Накопительный счет",
]

TOPICS = [
    "условия продукта",
    "оформление продукта",
    "комиссия",
    "ошибка в приложении",
    "блокировка карты",
    "просрочка платежа",
    "возврат средств",
    "повышение лимита",
    "закрытие продукта",
    "повторная покупка",
    "жалоба на обслуживание",
    "консультация по тарифам",
    "техническая ошибка",
    "изменение персональных данных",
]

INCOME_LEVELS = ["low", "medium", "high"]
GENDERS = ["женский", "мужской"]
SENTIMENTS = ["positive", "neutral", "negative"]

CITIES = [
    "Москва",
    "Санкт-Петербург",
    "Казань",
    "Новосибирск",
    "Екатеринбург",
    "Нижний Новгород",
    "Самара",
    "Ростов-на-Дону",
    "Краснодар",
    "Воронеж",
    "Пермь",
    "Уфа",
]

EVENT_TYPE_TO_STAGE = {
    "introduction": "знакомство",
    "product_interest": "интерес к продукту",
    "consultation": "консультация",
    "purchase": "покупка",
    "usage": "использование продукта",
    "support_question": "вопрос в поддержку",
    "complaint": "жалоба",
    "escalation": "эскалация",
    "problem_resolved": "решение проблемы",
    "retention": "удержание",
    "repeat_purchase": "повторная покупка",
    "product_closure": "закрытие продукта",
    "churn": "уход клиента",
}

TOPICS_BY_EVENT_TYPE = {
    "introduction": ["условия продукта", "консультация по тарифам", "оформление продукта"],
    "product_interest": ["условия продукта", "повышение лимита", "консультация по тарифам"],
    "consultation": ["консультация по тарифам", "условия продукта", "изменение персональных данных"],
    "purchase": ["оформление продукта", "условия продукта"],
    "usage": ["комиссия", "условия продукта", "изменение персональных данных"],
    "support_question": ["ошибка в приложении", "блокировка карты", "техническая ошибка", "изменение персональных данных"],
    "complaint": ["жалоба на обслуживание", "комиссия", "ошибка в приложении", "возврат средств", "блокировка карты", "просрочка платежа"],
    "escalation": ["жалоба на обслуживание", "возврат средств", "техническая ошибка", "блокировка карты"],
    "problem_resolved": ["возврат средств", "техническая ошибка", "блокировка карты", "жалоба на обслуживание"],
    "retention": ["условия продукта", "консультация по тарифам", "повышение лимита"],
    "repeat_purchase": ["повторная покупка", "оформление продукта", "условия продукта"],
    "product_closure": ["закрытие продукта", "комиссия"],
    "churn": ["закрытие продукта", "жалоба на обслуживание", "комиссия"],
}

LIFECYCLE_SCENARIOS = {
    "one_touch": ["introduction"],
    "short_support": ["introduction", "support_question", "problem_resolved"],
    "happy_purchase": [
        "introduction",
        "product_interest",
        "consultation",
        "purchase",
        "usage",
        "support_question",
        "problem_resolved",
    ],
    "complaint_and_leave": [
        "introduction",
        "purchase",
        "usage",
        "complaint",
        "escalation",
        "churn",
    ],
    "complaint_resolved_return": [
        "usage",
        "complaint",
        "escalation",
        "problem_resolved",
        "retention",
        "repeat_purchase",
    ],
    "loyal_multi_product": [
        "introduction",
        "product_interest",
        "purchase",
        "usage",
        "repeat_purchase",
        "usage",
        "support_question",
        "problem_resolved",
        "repeat_purchase",
        "retention",
    ],
    "problematic_many_complaints": [
        "usage",
        "complaint",
        "escalation",
        "problem_resolved",
        "complaint",
        "support_question",
        "complaint",
        "escalation",
        "product_closure",
        "churn",
    ],
    "long_lifecycle": [
        "introduction",
        "product_interest",
        "consultation",
        "purchase",
        "usage",
        "support_question",
        "problem_resolved",
        "retention",
        "repeat_purchase",
        "usage",
        "consultation",
        "repeat_purchase",
    ],
    "repeat_purchase": [
        "introduction",
        "product_interest",
        "purchase",
        "usage",
        "consultation",
        "repeat_purchase",
        "usage",
        "repeat_purchase",
    ],
}

SCENARIO_DESCRIPTIONS = {
    "one_touch": "клиент обратился один раз и больше не вернулся",
    "short_support": "короткий путь с вопросом в поддержку",
    "happy_purchase": "знакомство, консультация, покупка и поддержка",
    "complaint_and_leave": "жалоба, эскалация и уход после негативного опыта",
    "complaint_resolved_return": "жалоба решена, клиент удержан и вернулся",
    "loyal_multi_product": "лояльный клиент покупает несколько продуктов",
    "problematic_many_complaints": "частые жалобы и высокий риск ухода",
    "long_lifecycle": "длинный путь на несколько месяцев",
    "repeat_purchase": "повторные покупки после первого продукта",
}

SCENARIO_EXTENSIONS = {
    "loyal_multi_product": ["usage", "consultation", "repeat_purchase", "support_question", "problem_resolved"],
    "problematic_many_complaints": ["complaint", "escalation", "problem_resolved", "support_question"],
    "long_lifecycle": ["usage", "consultation", "repeat_purchase", "support_question", "problem_resolved"],
    "repeat_purchase": ["usage", "consultation", "repeat_purchase", "usage"],
    "happy_purchase": ["usage", "support_question", "problem_resolved"],
    "complaint_resolved_return": ["usage", "support_question", "repeat_purchase"],
    "short_support": ["support_question", "problem_resolved"],
    "complaint_and_leave": ["complaint", "escalation", "churn"],
}

NODE_COLORS = {
    "Client": "#2563eb",
    "Event": "#f97316",
    "Message": "#64748b",
    "Product": "#16a34a",
    "Topic": "#db2777",
    "LifecycleStage": "#7c3aed",
    "Scenario": "#0891b2",
    "Channel": "#ca8a04",
    "Sentiment": "#475569",
    "Segment": "#0d9488",
    "City": "#9333ea",
}

NODE_SHAPES = {
    "Client": "dot",
    "Event": "box",
    "Message": "ellipse",
    "Product": "diamond",
    "Topic": "dot",
    "LifecycleStage": "triangle",
    "Scenario": "star",
    "Channel": "square",
    "Sentiment": "dot",
    "Segment": "hexagon",
    "City": "dot",
}

EVENT_START_MIN = datetime(2024, 8, 1)
EVENT_START_MAX = datetime(2025, 1, 31)


def random_date(start_date, end_date):
    delta_days = (end_date - start_date).days
    return start_date + timedelta(days=random.randint(0, delta_days))


def weighted_choice(options, weights):
    return random.choices(options, weights=weights, k=1)[0]


def short_text(value, limit=42):
    value = str(value)
    return value if len(value) <= limit else value[: limit - 3] + "..."


def html_title(rows):
    safe_rows = []
    for key, value in rows:
        if value is None or (isinstance(value, float) and np.isnan(value)):
            continue
        safe_rows.append(f"<b>{html.escape(str(key))}</b>: {html.escape(str(value))}")
    return "<br>".join(safe_rows)

## 4. Генерация синтетических клиентов

Создается `clients_df` на 50 вымышленных клиентов с сегментами, каналами, уровнем дохода, риском оттока и базовыми продуктами.

In [4]:
client_rows = []

segment_weights = [10, 13, 7, 6, 7, 4, 3]
churn_risk_by_segment = {
    "new": ["low", "medium"],
    "regular": ["low", "medium"],
    "premium": ["low"],
    "risk": ["high", "critical"],
    "sleeping": ["medium", "high"],
    "problematic": ["high", "critical"],
    "loyal": ["low"],
}

for i in range(1, 51):
    client_id = f"C{i:03d}"
    gender = random.choice(GENDERS)
    client_name = fake.name_female() if gender == "женский" else fake.name_male()
    age = random.randint(18, 74)
    segment = weighted_choice(SEGMENTS, segment_weights)
    city = random.choice(CITIES)
    preferred_channel = random.choice(CHANNELS)
    income_level = weighted_choice(INCOME_LEVELS, [0.28, 0.52, 0.20])
    base_product_count = weighted_choice([1, 2, 3], [0.55, 0.35, 0.10])
    base_products = random.sample(PRODUCTS, k=base_product_count)

    if segment in ["premium", "loyal"]:
        loyalty_score = random.randint(75, 100)
    elif segment in ["risk", "problematic"]:
        loyalty_score = random.randint(15, 55)
    elif segment == "sleeping":
        loyalty_score = random.randint(30, 65)
    else:
        loyalty_score = random.randint(45, 85)

    created_at = random_date(datetime(2023, 1, 1), datetime(2024, 7, 1)).date().isoformat()

    client_rows.append(
        {
            "client_id": client_id,
            "client_name": client_name,
            "age": age,
            "gender": gender,
            "city": city,
            "segment": segment,
            "loyalty_score": loyalty_score,
            "churn_risk": random.choice(churn_risk_by_segment[segment]),
            "income_level": income_level,
            "preferred_channel": preferred_channel,
            "created_at": created_at,
            "base_products": base_products,
        }
    )

clients_df = pd.DataFrame(client_rows)
display(clients_df.head(10))
print("Количество клиентов:", len(clients_df))

,client_id,client_name,age,gender,city,segment,loyalty_score,churn_risk,income_level,preferred_channel,created_at,base_products
0,C001,Алевтина Игоревна Николаева,19,женский,Новосибирск,sleeping,64,high,low,сайт,2023-03-31,[Страхование]
1,C002,Зиновьева Валентина Тимофеевна,19,женский,Новосибирск,new,79,low,medium,чат,2024-03-05,"[Страхование, Накопительный счет]"
2,C003,Родионов Амос Аксёнович,55,мужской,Москва,regular,58,low,medium,сайт,2023-12-11,[Кредитная карта]
3,C004,Жукова Прасковья Юльевна,42,женский,Нижний Новгород,new,79,medium,low,чат,2023-05-08,[Ипотека]
4,C005,Лазарева София Николаевна,53,женский,Пермь,regular,49,low,high,чат,2023-02-16,[Кредитная карта]
5,C006,Сафонов Клавдий Абрамович,23,мужской,Санкт-Петербург,sleeping,53,medium,low,телефон,2023-12-30,"[Потребительский кредит, Кредитная карта]"
6,C007,Гордей Федосьевич Александров,62,мужской,Пермь,problematic,30,critical,medium,мобильное приложение,2023-06-17,[Страхование]
7,C008,Рябов Мартын Бориславович,35,мужской,Пермь,loyal,82,low,medium,email,2023-02-02,"[Накопительный счет, Дебетовая карта]"
8,C009,Белов Мирон Изотович,35,мужской,Воронеж,new,74,medium,medium,email,2023-05-27,"[Ипотека, Страхование]"
9,C010,Алина Робертовна Белякова,33,женский,Краснодар,sleeping,55,medium,medium,офис,2024-01-06,[Инвестиционный счет]


Количество клиентов: 50


## 5. Генерация разных жизненных циклов

В этой версии у клиентов не один общий путь, а разные сценарии и разная длина цепочки обращений.  
Распределение заранее задается так, чтобы гарантировать короткие, обычные и длинные пути.

In [5]:
def build_lifecycle_sequence(scenario_name, target_event_count):
    sequence = list(LIFECYCLE_SCENARIOS[scenario_name])

    if scenario_name == "complaint_and_leave" and len(sequence) > target_event_count:
        if target_event_count == 1:
            sequence = ["complaint"]
        elif target_event_count == 2:
            sequence = ["complaint", "churn"]
        elif target_event_count == 3:
            sequence = ["usage", "complaint", "churn"]
        else:
            sequence = sequence[: target_event_count - 1] + ["churn"]
    elif len(sequence) > target_event_count:
        sequence = sequence[:target_event_count]

    extension_pool = SCENARIO_EXTENSIONS.get(scenario_name, ["usage", "support_question"])
    while len(sequence) < target_event_count:
        sequence.append(random.choice(extension_pool))

    return sequence[:target_event_count]


scenario_plan = []

# Минимум 8 клиентов с одним обращением.
for _ in range(8):
    scenario_plan.append({"scenario_name": "one_touch", "target_event_count": 1, "length_group": "1 обращение"})

# Минимум 10 клиентов с 2-3 обращениями.
short_scenarios = ["short_support", "complaint_and_leave", "happy_purchase"]
for _ in range(12):
    scenario_plan.append(
        {
            "scenario_name": random.choice(short_scenarios),
            "target_event_count": random.choice([2, 3]),
            "length_group": "2-3 обращения",
        }
    )

# Минимум 15 клиентов с обычным путем 5-7 обращений.
medium_scenarios = [
    "happy_purchase",
    "complaint_resolved_return",
    "complaint_and_leave",
    "loyal_multi_product",
    "repeat_purchase",
]
for _ in range(22):
    scenario_plan.append(
        {
            "scenario_name": random.choice(medium_scenarios),
            "target_event_count": random.choice([5, 6, 7]),
            "length_group": "5-7 обращений",
        }
    )

# Минимум 5 клиентов с длинным путем 8+ обращений.
long_scenarios = ["long_lifecycle", "loyal_multi_product", "problematic_many_complaints", "repeat_purchase"]
for _ in range(8):
    scenario_plan.append(
        {
            "scenario_name": random.choice(long_scenarios),
            "target_event_count": random.randint(8, 12),
            "length_group": "8+ обращений",
        }
    )

random.shuffle(scenario_plan)

lifecycle_plan_rows = []
for client_id, plan_item in zip(clients_df["client_id"], scenario_plan):
    sequence = build_lifecycle_sequence(plan_item["scenario_name"], plan_item["target_event_count"])
    lifecycle_plan_rows.append(
        {
            "client_id": client_id,
            "scenario_name": plan_item["scenario_name"],
            "scenario_description": SCENARIO_DESCRIPTIONS[plan_item["scenario_name"]],
            "target_event_count": plan_item["target_event_count"],
            "length_group": plan_item["length_group"],
            "event_sequence": sequence,
            "path_signature": " > ".join(sequence),
        }
    )

lifecycle_plan_df = pd.DataFrame(lifecycle_plan_rows)

display(lifecycle_plan_df.head(10))
display(
    lifecycle_plan_df.groupby(["length_group", "scenario_name"], as_index=False)
    .size()
    .sort_values(["length_group", "scenario_name"])
)

,client_id,scenario_name,scenario_description,target_event_count,length_group,event_sequence,path_signature
0,C001,problematic_many_complaints,частые жалобы и высокий риск ухода,8,8+ обращений,"[usage, complaint, escalation, problem_resolved, complaint, support_question, complaint, escalation]",usage > complaint > escalation > problem_resolved > complaint > support_question > complaint > escalation
1,C002,complaint_resolved_return,"жалоба решена, клиент удержан и вернулся",5,5-7 обращений,"[usage, complaint, escalation, problem_resolved, retention]",usage > complaint > escalation > problem_resolved > retention
2,C003,complaint_and_leave,"жалоба, эскалация и уход после негативного опыта",3,2-3 обращения,"[usage, complaint, churn]",usage > complaint > churn
3,C004,one_touch,клиент обратился один раз и больше не вернулся,1,1 обращение,[introduction],introduction
4,C005,loyal_multi_product,лояльный клиент покупает несколько продуктов,5,5-7 обращений,"[introduction, product_interest, purchase, usage, repeat_purchase]",introduction > product_interest > purchase > usage > repeat_purchase
5,C006,complaint_and_leave,"жалоба, эскалация и уход после негативного опыта",2,2-3 обращения,"[complaint, churn]",complaint > churn
6,C007,complaint_and_leave,"жалоба, эскалация и уход после негативного опыта",6,5-7 обращений,"[introduction, purchase, usage, complaint, escalation, churn]",introduction > purchase > usage > complaint > escalation > churn
7,C008,complaint_and_leave,"жалоба, эскалация и уход после негативного опыта",5,5-7 обращений,"[introduction, purchase, usage, complaint, churn]",introduction > purchase > usage > complaint > churn
8,C009,complaint_resolved_return,"жалоба решена, клиент удержан и вернулся",6,5-7 обращений,"[usage, complaint, escalation, problem_resolved, retention, repeat_purchase]",usage > complaint > escalation > problem_resolved > retention > repeat_purchase
9,C010,happy_purchase,"знакомство, консультация, покупка и поддержка",7,5-7 обращений,"[introduction, product_interest, consultation, purchase, usage, support_question, problem_resolved]",introduction > product_interest > consultation > purchase > usage > support_question > problem_resolved


,length_group,scenario_name,size
0,1 обращение,one_touch,8
1,2-3 обращения,complaint_and_leave,8
2,2-3 обращения,happy_purchase,3
3,2-3 обращения,short_support,1
4,5-7 обращений,complaint_and_leave,7
5,5-7 обращений,complaint_resolved_return,7
6,5-7 обращений,happy_purchase,4
7,5-7 обращений,loyal_multi_product,2
8,5-7 обращений,repeat_purchase,2
9,8+ обращений,long_lifecycle,3


## 6. Генерация обращений клиентов

Создается `events_df`. Даты обращений идут по порядку, между событиями проходит от 7 до 45 дней, а `previous_event_id` связывает событие с предыдущим обращением клиента.

In [6]:
def choose_topic(event_type):
    return random.choice(TOPICS_BY_EVENT_TYPE[event_type])


def choose_product_for_event(event_type, owned_products):
    if event_type in ["purchase", "repeat_purchase"]:
        available_products = [product for product in PRODUCTS if product not in owned_products]
        product = random.choice(available_products or PRODUCTS)
        if product not in owned_products:
            owned_products.append(product)
        return product

    if event_type in ["usage", "support_question", "complaint", "escalation", "problem_resolved", "product_closure", "churn"]:
        if owned_products and random.random() < 0.82:
            return random.choice(owned_products)
        return random.choice(PRODUCTS)

    if owned_products and random.random() < 0.45:
        return random.choice(owned_products)
    return random.choice(PRODUCTS)


def generate_sentiment_score_status(event_type):
    if event_type == "complaint":
        sentiment = weighted_choice(SENTIMENTS, [0.05, 0.20, 0.75])
        satisfaction_score = random.randint(1, 3)
        status = weighted_choice(["in_progress", "escalated", "resolved"], [0.45, 0.40, 0.15])
        resolution_time_days = random.randint(3, 14) if status == "resolved" else None
    elif event_type == "escalation":
        sentiment = weighted_choice(SENTIMENTS, [0.03, 0.17, 0.80])
        satisfaction_score = random.randint(1, 3)
        status = "escalated"
        resolution_time_days = None
    elif event_type == "problem_resolved":
        sentiment = weighted_choice(SENTIMENTS, [0.68, 0.27, 0.05])
        satisfaction_score = random.randint(4, 5)
        status = "resolved"
        resolution_time_days = random.randint(1, 5)
    elif event_type == "churn":
        sentiment = weighted_choice(SENTIMENTS, [0.02, 0.13, 0.85])
        satisfaction_score = random.randint(1, 2)
        status = "lost"
        resolution_time_days = None
    elif event_type in ["purchase", "repeat_purchase"]:
        sentiment = weighted_choice(SENTIMENTS, [0.78, 0.18, 0.04])
        satisfaction_score = random.randint(4, 5)
        status = "completed"
        resolution_time_days = random.randint(0, 2)
    elif event_type == "support_question":
        sentiment = weighted_choice(SENTIMENTS, [0.25, 0.55, 0.20])
        satisfaction_score = random.randint(3, 5) if sentiment != "negative" else random.randint(2, 3)
        status = weighted_choice(["resolved", "in_progress"], [0.70, 0.30])
        resolution_time_days = random.randint(1, 7) if status == "resolved" else None
    elif event_type == "product_closure":
        sentiment = weighted_choice(SENTIMENTS, [0.08, 0.42, 0.50])
        satisfaction_score = random.randint(2, 4)
        status = "closed"
        resolution_time_days = random.randint(0, 3)
    elif event_type == "retention":
        sentiment = weighted_choice(SENTIMENTS, [0.58, 0.35, 0.07])
        satisfaction_score = random.randint(3, 5)
        status = "retained"
        resolution_time_days = random.randint(0, 4)
    else:
        sentiment = weighted_choice(SENTIMENTS, [0.42, 0.50, 0.08])
        satisfaction_score = random.randint(3, 5)
        status = "completed"
        resolution_time_days = random.randint(0, 3)

    return sentiment, status, satisfaction_score, resolution_time_days


event_rows = []

for plan_row in lifecycle_plan_df.itertuples(index=False):
    client = clients_df.loc[clients_df["client_id"] == plan_row.client_id].iloc[0]
    owned_products = list(client["base_products"])
    current_date = random_date(EVENT_START_MIN, EVENT_START_MAX)
    previous_event_id = None

    for event_order, event_type in enumerate(plan_row.event_sequence, start=1):
        if event_order > 1:
            current_date = current_date + timedelta(days=random.randint(7, 45))

        product = choose_product_for_event(event_type, owned_products)
        topic = choose_topic(event_type)
        sentiment, status, satisfaction_score, resolution_time_days = generate_sentiment_score_status(event_type)
        lifecycle_stage = EVENT_TYPE_TO_STAGE[event_type]
        event_id = "event_" + uuid.uuid5(
            uuid.NAMESPACE_DNS,
            f"{plan_row.client_id}-{event_order}-{event_type}-{product}-{topic}",
        ).hex[:12]

        event_rows.append(
            {
                "event_id": event_id,
                "client_id": plan_row.client_id,
                "event_order": event_order,
                "previous_event_id": previous_event_id,
                "event_date": pd.Timestamp(current_date.date()),
                "event_month": current_date.strftime("%Y-%m"),
                "scenario_name": plan_row.scenario_name,
                "lifecycle_stage": lifecycle_stage,
                "event_type": event_type,
                "product": product,
                "topic": topic,
                "channel": random.choice([client["preferred_channel"], random.choice(CHANNELS)]),
                "sentiment": sentiment,
                "status": status,
                "satisfaction_score": satisfaction_score,
                "is_complaint": event_type in ["complaint", "escalation"],
                "is_purchase": event_type in ["purchase", "repeat_purchase"],
                "is_repeat": event_order > 1,
                "resolution_time_days": resolution_time_days,
            }
        )

        previous_event_id = event_id

        if event_type in ["product_closure", "churn"] and product in owned_products and len(owned_products) > 1:
            owned_products.remove(product)

events_df = (
    pd.DataFrame(event_rows)
    .sort_values(["client_id", "event_order"])
    .reset_index(drop=True)
)

display(events_df.head(12))

event_count_distribution = (
    events_df.groupby("client_id").size().value_counts().sort_index().rename_axis("event_count").reset_index(name="clients")
)
display(event_count_distribution)
print("Всего обращений:", len(events_df))

,event_id,client_id,event_order,previous_event_id,event_date,event_month,scenario_name,lifecycle_stage,event_type,product,topic,channel,sentiment,status,satisfaction_score,is_complaint,is_purchase,is_repeat,resolution_time_days
0,event_27771ad1ae68,C001,1,None,2024-12-11,2024-12,problematic_many_complaints,использование продукта,usage,Дебетовая карта,комиссия,email,neutral,completed,4,False,False,False,1.0
1,event_23cb234e9bed,C001,2,event_27771ad1ae68,2025-01-18,2025-01,problematic_many_complaints,жалоба,complaint,Страхование,жалоба на обслуживание,офис,negative,escalated,2,True,False,True,NaN
2,event_8c29135abac0,C001,3,event_23cb234e9bed,2025-02-24,2025-02,problematic_many_complaints,эскалация,escalation,Страхование,блокировка карты,офис,negative,escalated,3,True,False,True,NaN
3,event_65d9b172fbc2,C001,4,event_8c29135abac0,2025-04-01,2025-04,problematic_many_complaints,решение проблемы,problem_resolved,Страхование,техническая ошибка,сайт,positive,resolved,4,False,False,True,3.0
4,event_13fa3c53a9e9,C001,5,event_65d9b172fbc2,2025-04-20,2025-04,problematic_many_complaints,жалоба,complaint,Страхование,ошибка в приложении,сайт,negative,escalated,3,True,False,True,NaN
5,event_9574cad4294c,C001,6,event_13fa3c53a9e9,2025-05-15,2025-05,problematic_many_complaints,вопрос в поддержку,support_question,Страхование,техническая ошибка,мобильное приложение,positive,resolved,5,False,False,True,1.0
6,event_083e4c73eb20,C001,7,event_9574cad4294c,2025-05-30,2025-05,problematic_many_complaints,жалоба,complaint,Страхование,жалоба на обслуживание,телефон,negative,in_progress,3,True,False,True,NaN
7,event_27f9b4ebb9c8,C001,8,event_083e4c73eb20,2025-06-27,2025-06,problematic_many_complaints,эскалация,escalation,Страхование,техническая ошибка,сайт,negative,escalated,2,True,False,True,NaN
8,event_0d8e4d9e5b33,C002,1,None,2024-11-11,2024-11,complaint_resolved_return,использование продукта,usage,Страхование,комиссия,чат,positive,completed,5,False,False,False,2.0
9,event_33213a837f9a,C002,2,event_0d8e4d9e5b33,2024-11-25,2024-11,complaint_resolved_return,жалоба,complaint,Накопительный счет,блокировка карты,чат,negative,in_progress,3,True,False,True,NaN


,event_count,clients
0,1,8
1,2,7
2,3,5
3,5,9
4,6,8
5,7,5
6,8,3
7,9,1
8,10,1
9,11,1


Всего обращений: 243


## 7. Генерация синтетических диалогов

Для каждого обращения генерируется 4-8 сообщений на русском языке. Диалоги зависят от типа события, продукта, темы и настроения.

In [7]:
CLIENT_OPENERS = {
    "introduction": [
        "Здравствуйте, хочу понять, как у вас работает продукт «{product}». Интересуют {topic}.",
        "Добрый день. Я впервые обращаюсь в банк и хочу узнать про «{product}».",
        "Здравствуйте. Подскажите, пожалуйста, что важно знать про «{product}»?",
    ],
    "product_interest": [
        "Рассматриваю «{product}» и хочу уточнить условия перед оформлением.",
        "Мне интересен продукт «{product}». Можете рассказать подробнее про {topic}?",
        "Хочу сравнить условия по продукту «{product}» с текущими вариантами.",
    ],
    "consultation": [
        "Нужна консультация по продукту «{product}», особенно по теме «{topic}».",
        "Помогите разобраться, подойдет ли мне «{product}».",
        "Хочу уточнить детали по тарифам и условиям для «{product}».",
    ],
    "purchase": [
        "Хочу оформить «{product}». Подскажите, что нужно сделать.",
        "Готов подключить продукт «{product}», хочу пройти оформление.",
        "Я принял решение оформить «{product}». Помогите завершить заявку.",
    ],
    "usage": [
        "Пользуюсь продуктом «{product}» и хочу уточнить вопрос: {topic}.",
        "У меня есть вопрос по использованию продукта «{product}».",
        "Подскажите, пожалуйста, как правильно действовать по теме «{topic}».",
    ],
    "support_question": [
        "Нужна помощь по продукту «{product}»: возник вопрос «{topic}».",
        "Не получается разобраться с ситуацией по продукту «{product}».",
        "Обращаюсь в поддержку, потому что есть проблема: {topic}.",
    ],
    "complaint": [
        "Я недоволен ситуацией по продукту «{product}». Проблема: {topic}.",
        "Хочу оставить жалобу. По продукту «{product}» возникла неприятная ситуация.",
        "Меня не устраивает качество обслуживания. Тема обращения: {topic}.",
    ],
    "escalation": [
        "Прошу передать обращение старшему специалисту. Вопрос по «{product}» не решен.",
        "Ситуация по теме «{topic}» затянулась, нужна эскалация.",
        "Я уже обращался, но проблема с продуктом «{product}» остается.",
    ],
    "problem_resolved": [
        "Проверяю статус проблемы по продукту «{product}». Есть ли решение?",
        "Возвращаюсь по обращению «{topic}». Хотелось бы понять результат.",
        "Мне сообщили, что вопрос почти решен. Подтвердите, пожалуйста.",
    ],
    "retention": [
        "После прошлой ситуации хочу понять, какие условия вы можете предложить.",
        "Я готов остаться клиентом, если вопрос по «{product}» действительно закрыт.",
        "Рассматриваю возможность продолжить пользоваться банком.",
    ],
    "repeat_purchase": [
        "Я уже клиент банка и хочу оформить еще один продукт: «{product}».",
        "После предыдущего опыта хочу рассмотреть повторную покупку продукта «{product}».",
        "Интересует дополнительный продукт «{product}». Расскажите про оформление.",
    ],
    "product_closure": [
        "Хочу закрыть продукт «{product}». Подскажите порядок действий.",
        "Планирую отказаться от продукта «{product}».",
        "Мне нужно закрыть продукт, причина связана с темой «{topic}».",
    ],
    "churn": [
        "Я решил прекратить обслуживание. Прошу закрыть все вопросы по «{product}».",
        "После последнего опыта я хочу уйти из банка.",
        "Не вижу смысла продолжать обслуживание, хочу завершить отношения с банком.",
    ],
}

OPERATOR_RESPONSES = {
    "introduction": [
        "Здравствуйте. Расскажу основные условия и помогу выбрать подходящий вариант.",
        "Добрый день. Сейчас поясню, как устроен продукт и какие шаги нужны.",
        "Здравствуйте. Я помогу сориентироваться по условиям и ограничениям.",
    ],
    "product_interest": [
        "Понимаю. Давайте уточним ваши цели и подберем подходящие условия.",
        "Могу рассказать про тариф, лимиты, комиссии и порядок оформления.",
        "Сейчас проверю актуальные условия для выбранного продукта.",
    ],
    "consultation": [
        "Конечно, давайте разберем условия и возможные ограничения.",
        "Я уточню параметры и объясню, какой вариант подойдет лучше.",
        "Сейчас отвечу по тарифам и помогу выбрать следующий шаг.",
    ],
    "purchase": [
        "Отлично, я помогу оформить заявку и проверю необходимые данные.",
        "Для оформления понадобится подтвердить данные и выбрать параметры продукта.",
        "Запускаю процесс оформления. Я буду сопровождать вас на каждом шаге.",
    ],
    "usage": [
        "Сейчас проверю информацию по продукту и объясню порядок действий.",
        "Давайте посмотрим детали операции и найдем ответ.",
        "Я помогу разобраться с использованием продукта.",
    ],
    "support_question": [
        "Понимаю. Сейчас проверю обращение и предложу решение.",
        "Давайте уточним детали проблемы, чтобы быстрее помочь.",
        "Я зафиксировал вопрос и проверяю доступные действия.",
    ],
    "complaint": [
        "Мне жаль, что вы столкнулись с такой ситуацией. Я зарегистрирую жалобу.",
        "Понимаю ваше недовольство. Зафиксирую обращение и передам его в работу.",
        "Давайте подробно опишем проблему, чтобы ее можно было проверить.",
    ],
    "escalation": [
        "Я передаю обращение на следующий уровень поддержки и отмечаю срочность.",
        "Эскалация оформлена. Специалист проверит историю обращений.",
        "Понимаю. Подключаю профильную команду для дополнительной проверки.",
    ],
    "problem_resolved": [
        "Проверила статус: решение уже применено, сейчас объясню детали.",
        "По вашему обращению есть результат. Проблема закрыта, данные обновлены.",
        "Вопрос решен. Я расскажу, что было сделано и как проверить результат.",
    ],
    "retention": [
        "Мы ценим, что вы готовы продолжить обслуживание. Предложу варианты удержания.",
        "Понимаю ваши сомнения. Давайте подберем условия, которые исправят опыт.",
        "Могу предложить индивидуальные условия и контроль следующего обращения.",
    ],
    "repeat_purchase": [
        "Рада, что вы вернулись. Проверю доступные условия для нового продукта.",
        "Давайте оформим дополнительный продукт и учтем ваш текущий профиль.",
        "Сейчас подберу вариант с учетом уже подключенных продуктов.",
    ],
    "product_closure": [
        "Понимаю. Проверю, нет ли задолженности или незавершенных операций.",
        "Для закрытия продукта нужно пройти короткую проверку статуса.",
        "Я помогу закрыть продукт и объясню, когда изменения вступят в силу.",
    ],
    "churn": [
        "Мне жаль, что вы приняли такое решение. Я помогу корректно завершить обслуживание.",
        "Понимаю. Зафиксирую причину ухода и проверю, что нужно закрыть.",
        "Сейчас оформим завершение обслуживания и подтвердим финальный статус.",
    ],
}

CLIENT_FOLLOW_UPS = [
    "Важно, чтобы решение было понятным и без скрытых условий.",
    "Я хочу понимать сроки и что будет происходить дальше.",
    "Уточните, пожалуйста, это повлияет на мои текущие продукты?",
    "Можно ли получить подтверждение после завершения операции?",
    "Мне удобнее продолжить общение через канал «{channel}».",
]

OPERATOR_FINALS = [
    "Я зафиксировал обращение. Статус сейчас: {status}.",
    "Информация передана в работу, ориентировочный срок решения зависит от проверки.",
    "По этому обращению указан статус «{status}», оценка удовлетворенности будет учтена.",
    "Отправлю вам подтверждение и краткое резюме по теме «{topic}».",
    "Если появятся дополнительные вопросы, можно продолжить диалог в этом же канале.",
]

SYSTEM_MESSAGES = [
    "Системное уведомление: обращение зарегистрировано и связано с продуктом «{product}».",
    "Системное уведомление: клиенту присвоен статус обращения «{status}».",
    "Системное уведомление: тема обращения классифицирована как «{topic}».",
    "Системное уведомление: история взаимодействий клиента обновлена.",
]


def render_dialog_template(template, event_row, client_name):
    return template.format(
        product=event_row["product"],
        topic=event_row["topic"],
        channel=event_row["channel"],
        status=event_row["status"],
        client_name=client_name,
        sentiment=event_row["sentiment"],
    )


def generate_dialog_for_event(event_row, client_name):
    message_count = random.randint(4, 8)
    event_type = event_row["event_type"]

    core_messages = [
        ("client", render_dialog_template(random.choice(CLIENT_OPENERS[event_type]), event_row, client_name)),
        ("operator", render_dialog_template(random.choice(OPERATOR_RESPONSES[event_type]), event_row, client_name)),
        ("client", render_dialog_template(random.choice(CLIENT_FOLLOW_UPS), event_row, client_name)),
        ("operator", render_dialog_template(random.choice(OPERATOR_FINALS), event_row, client_name)),
    ]

    optional_messages = [
        ("system", render_dialog_template(random.choice(SYSTEM_MESSAGES), event_row, client_name)),
        ("client", f"Меня зовут {client_name}. Прошу учесть мой прошлый опыт и текущий продукт."),
        ("operator", f"Я вижу историю клиента и учитываю этап «{event_row['lifecycle_stage']}»."),
        ("system", f"Системное уведомление: настроение обращения определено как «{event_row['sentiment']}»."),
    ]

    random.shuffle(optional_messages)
    selected_optional = optional_messages[: max(0, message_count - 4)]
    message_flow = core_messages[:3] + selected_optional + core_messages[3:]

    return message_flow[:message_count]


dialog_rows = []

clients_lookup = clients_df.set_index("client_id")["client_name"].to_dict()

for event_row in events_df.to_dict("records"):
    client_name = clients_lookup[event_row["client_id"]]
    for message_order, (speaker, text) in enumerate(generate_dialog_for_event(event_row, client_name), start=1):
        message_id = "message_" + uuid.uuid5(
            uuid.NAMESPACE_DNS,
            f"{event_row['event_id']}-{message_order}-{speaker}-{text}",
        ).hex[:12]

        dialog_rows.append(
            {
                "message_id": message_id,
                "event_id": event_row["event_id"],
                "client_id": event_row["client_id"],
                "message_order": message_order,
                "speaker": speaker,
                "text": text,
                "event_type": event_row["event_type"],
                "lifecycle_stage": event_row["lifecycle_stage"],
                "product": event_row["product"],
                "topic": event_row["topic"],
                "sentiment": event_row["sentiment"],
                "event_date": event_row["event_date"],
            }
        )

dialogs_df = pd.DataFrame(dialog_rows).sort_values(["client_id", "event_id", "message_order"]).reset_index(drop=True)

display(dialogs_df.head(15))
print("Всего сообщений:", len(dialogs_df))
print("Сообщений на обращение: от", dialogs_df.groupby("event_id").size().min(), "до", dialogs_df.groupby("event_id").size().max())

,message_id,event_id,client_id,message_order,speaker,text,event_type,lifecycle_stage,product,topic,sentiment,event_date
0,message_508b92b479f9,event_083e4c73eb20,C001,1,client,Меня не устраивает качество обслуживания. Тема обращения: жалоба на обслуживание.,complaint,жалоба,Страхование,жалоба на обслуживание,negative,2025-05-30
1,message_11fb6424a80c,event_083e4c73eb20,C001,2,operator,"Мне жаль, что вы столкнулись с такой ситуацией. Я зарегистрирую жалобу.",complaint,жалоба,Страхование,жалоба на обслуживание,negative,2025-05-30
2,message_9c2a654035e8,event_083e4c73eb20,C001,3,client,Можно ли получить подтверждение после завершения операции?,complaint,жалоба,Страхование,жалоба на обслуживание,negative,2025-05-30
3,message_e1157375f8ab,event_083e4c73eb20,C001,4,system,Системное уведомление: настроение обращения определено как «negative».,complaint,жалоба,Страхование,жалоба на обслуживание,negative,2025-05-30
4,message_32feb61e6110,event_083e4c73eb20,C001,5,client,Меня зовут Алевтина Игоревна Николаева. Прошу учесть мой прошлый опыт и текущий продукт.,complaint,жалоба,Страхование,жалоба на обслуживание,negative,2025-05-30
5,message_d6dc7514be67,event_083e4c73eb20,C001,6,system,Системное уведомление: клиенту присвоен статус обращения «in_progress».,complaint,жалоба,Страхование,жалоба на обслуживание,negative,2025-05-30
6,message_5c80086c7fdd,event_083e4c73eb20,C001,7,operator,Я зафиксировал обращение. Статус сейчас: in_progress.,complaint,жалоба,Страхование,жалоба на обслуживание,negative,2025-05-30
7,message_43e1ab93d089,event_13fa3c53a9e9,C001,1,client,Хочу оставить жалобу. По продукту «Страхование» возникла неприятная ситуация.,complaint,жалоба,Страхование,ошибка в приложении,negative,2025-04-20
8,message_c29919ec37f6,event_13fa3c53a9e9,C001,2,operator,"Мне жаль, что вы столкнулись с такой ситуацией. Я зарегистрирую жалобу.",complaint,жалоба,Страхование,ошибка в приложении,negative,2025-04-20
9,message_e9f8ba430656,event_13fa3c53a9e9,C001,3,client,Мне удобнее продолжить общение через канал «сайт».,complaint,жалоба,Страхование,ошибка в приложении,negative,2025-04-20


Всего сообщений: 1444
Сообщений на обращение: от 4 до 8


## 8. Построение графа знаний

Граф строится через `networkx.DiGraph()`. Он связывает клиентов, обращения, сообщения, продукты, темы, этапы жизненного цикла, сценарии, каналы, настроения, сегменты и города.

In [8]:
G = nx.DiGraph()


def add_node_once(node_id, node_type, label, **attrs):
    node_attrs = {
        "node_id": node_id,
        "node_type": node_type,
        "label": label,
    }
    node_attrs.update(attrs)
    if node_id not in G:
        G.add_node(node_id, **node_attrs)
    else:
        G.nodes[node_id].update({k: v for k, v in node_attrs.items() if v is not None})


def add_relation(source, target, relation, **attrs):
    edge_attrs = {
        "source": source,
        "target": target,
        "relation": relation,
    }
    edge_attrs.update(attrs)
    G.add_edge(source, target, **edge_attrs)


for client in clients_df.to_dict("records"):
    client_node_id = f"Client::{client['client_id']}"
    add_node_once(
        client_node_id,
        "Client",
        f"{client['client_id']} | {client['client_name']}",
        client_id=client["client_id"],
        client_name=client["client_name"],
        age=client["age"],
        segment=client["segment"],
        loyalty_score=client["loyalty_score"],
        churn_risk=client["churn_risk"],
        city=client["city"],
    )

    segment_node_id = f"Segment::{client['segment']}"
    add_node_once(segment_node_id, "Segment", client["segment"], segment=client["segment"])
    add_relation(client_node_id, segment_node_id, "CLIENT_IN_SEGMENT")

    city_node_id = f"City::{client['city']}"
    add_node_once(city_node_id, "City", client["city"], city=client["city"])
    add_relation(client_node_id, city_node_id, "CLIENT_FROM_CITY")

    scenario_name = lifecycle_plan_df.loc[lifecycle_plan_df["client_id"] == client["client_id"], "scenario_name"].iloc[0]
    scenario_node_id = f"Scenario::{scenario_name}"
    add_node_once(
        scenario_node_id,
        "Scenario",
        scenario_name,
        scenario_name=scenario_name,
        description=SCENARIO_DESCRIPTIONS[scenario_name],
    )
    add_relation(client_node_id, scenario_node_id, "CLIENT_IN_SCENARIO")


for event in events_df.to_dict("records"):
    client_node_id = f"Client::{event['client_id']}"
    event_node_id = f"Event::{event['event_id']}"
    add_node_once(
        event_node_id,
        "Event",
        f"{event['event_order']}. {event['lifecycle_stage']}",
        event_id=event["event_id"],
        client_id=event["client_id"],
        event_order=event["event_order"],
        event_date=str(event["event_date"].date()),
        scenario_name=event["scenario_name"],
        lifecycle_stage=event["lifecycle_stage"],
        event_type=event["event_type"],
        product=event["product"],
        topic=event["topic"],
        channel=event["channel"],
        sentiment=event["sentiment"],
        status=event["status"],
        satisfaction_score=event["satisfaction_score"],
    )
    add_relation(client_node_id, event_node_id, "CLIENT_HAS_EVENT")

    if event["previous_event_id"] is not None:
        previous_event_node_id = f"Event::{event['previous_event_id']}"
        add_relation(previous_event_node_id, event_node_id, "EVENT_NEXT")

    product_node_id = f"Product::{event['product']}"
    add_node_once(product_node_id, "Product", event["product"], product=event["product"])
    add_relation(event_node_id, product_node_id, "EVENT_ABOUT_PRODUCT")

    topic_node_id = f"Topic::{event['topic']}"
    add_node_once(topic_node_id, "Topic", event["topic"], topic=event["topic"])
    add_relation(event_node_id, topic_node_id, "EVENT_HAS_TOPIC")

    stage_node_id = f"LifecycleStage::{event['lifecycle_stage']}"
    add_node_once(stage_node_id, "LifecycleStage", event["lifecycle_stage"], lifecycle_stage=event["lifecycle_stage"])
    add_relation(event_node_id, stage_node_id, "EVENT_HAS_STAGE")

    channel_node_id = f"Channel::{event['channel']}"
    add_node_once(channel_node_id, "Channel", event["channel"], channel=event["channel"])
    add_relation(event_node_id, channel_node_id, "EVENT_FROM_CHANNEL")

    sentiment_node_id = f"Sentiment::{event['sentiment']}"
    add_node_once(sentiment_node_id, "Sentiment", event["sentiment"], sentiment=event["sentiment"])
    add_relation(event_node_id, sentiment_node_id, "EVENT_HAS_SENTIMENT")


for message in dialogs_df.to_dict("records"):
    event_node_id = f"Event::{message['event_id']}"
    message_node_id = f"Message::{message['message_id']}"
    add_node_once(
        message_node_id,
        "Message",
        f"{message['speaker']}: {short_text(message['text'], 55)}",
        message_id=message["message_id"],
        event_id=message["event_id"],
        client_id=message["client_id"],
        speaker=message["speaker"],
        text=message["text"],
        event_date=str(pd.Timestamp(message["event_date"]).date()),
    )
    add_relation(event_node_id, message_node_id, "EVENT_HAS_MESSAGE")

nodes_df = pd.DataFrame([attrs for _, attrs in G.nodes(data=True)]).sort_values(["node_type", "node_id"]).reset_index(drop=True)
edges_df = pd.DataFrame([attrs for _, _, attrs in G.edges(data=True)]).sort_values(["relation", "source", "target"]).reset_index(drop=True)

print("Узлов в графе:", G.number_of_nodes())
print("Ребер в графе:", G.number_of_edges())
display(nodes_df.head(10))
display(edges_df.head(10))

Узлов в графе: 1808
Ребер в графе: 3245


,node_id,node_type,label,client_id,client_name,age,segment,loyalty_score,churn_risk,city,scenario_name,description,event_id,event_order,event_date,lifecycle_stage,event_type,product,topic,channel,sentiment,status,satisfaction_score,message_id,speaker,text
0,Channel::email,Channel,email,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,email,NaN,NaN,NaN,NaN,NaN,NaN
1,Channel::мобильное приложение,Channel,мобильное приложение,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,мобильное приложение,NaN,NaN,NaN,NaN,NaN,NaN
2,Channel::офис,Channel,офис,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,офис,NaN,NaN,NaN,NaN,NaN,NaN
3,Channel::сайт,Channel,сайт,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,сайт,NaN,NaN,NaN,NaN,NaN,NaN
4,Channel::телефон,Channel,телефон,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,телефон,NaN,NaN,NaN,NaN,NaN,NaN
5,Channel::чат,Channel,чат,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,чат,NaN,NaN,NaN,NaN,NaN,NaN
6,City::Воронеж,City,Воронеж,NaN,NaN,NaN,NaN,NaN,NaN,Воронеж,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,City::Екатеринбург,City,Екатеринбург,NaN,NaN,NaN,NaN,NaN,NaN,Екатеринбург,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,City::Казань,City,Казань,NaN,NaN,NaN,NaN,NaN,NaN,Казань,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,City::Краснодар,City,Краснодар,NaN,NaN,NaN,NaN,NaN,NaN,Краснодар,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,source,target,relation
0,Client::C001,City::Новосибирск,CLIENT_FROM_CITY
1,Client::C002,City::Новосибирск,CLIENT_FROM_CITY
2,Client::C003,City::Москва,CLIENT_FROM_CITY
3,Client::C004,City::Нижний Новгород,CLIENT_FROM_CITY
4,Client::C005,City::Пермь,CLIENT_FROM_CITY
5,Client::C006,City::Санкт-Петербург,CLIENT_FROM_CITY
6,Client::C007,City::Пермь,CLIENT_FROM_CITY
7,Client::C008,City::Пермь,CLIENT_FROM_CITY
8,Client::C009,City::Воронеж,CLIENT_FROM_CITY
9,Client::C010,City::Краснодар,CLIENT_FROM_CITY


## 9. Интерактивный граф жизненного пути одного клиента

Функция `show_client_lifecycle_graph(client_id)` показывает клиента, все его обращения, последовательность `EVENT_NEXT`, продукты, темы, этапы и первые сообщения по каждому событию.

In [9]:
def make_network(height="650px", width="100%"):
    net = Network(
        height=height,
        width=width,
        directed=True,
        notebook=False,
        bgcolor="#ffffff",
        font_color="#111827",
        cdn_resources="in_line",
    )
    net.barnes_hut(
        gravity=-24000,
        central_gravity=0.25,
        spring_length=120,
        spring_strength=0.02,
        damping=0.12,
    )
    return net


def add_pyvis_node(net, node_id, label, node_type, title="", size=18):
    net.add_node(
        node_id,
        label=short_text(label, 34),
        title=title,
        color=NODE_COLORS.get(node_type, "#94a3b8"),
        shape=NODE_SHAPES.get(node_type, "dot"),
        size=size,
    )


def render_pyvis_network(net, output_file, show=True, height=650):
    output_path = OUTPUT_DIR / output_file
    html_content = net.generate_html(notebook=False)
    output_path.write_text(html_content, encoding="utf-8")
    if show:
        display(IFrame(src=output_path.as_posix(), width="100%", height=height))
    return output_path


def show_client_lifecycle_graph(client_id, output_file=None, show=True, max_messages_per_event=2):
    client_row = clients_df.loc[clients_df["client_id"] == client_id].iloc[0]
    client_events = events_df.loc[events_df["client_id"] == client_id].sort_values("event_order")
    client_messages = dialogs_df.loc[dialogs_df["client_id"] == client_id]

    if output_file is None:
        output_file = f"client_lifecycle_{client_id}.html"

    net = make_network()

    client_node_id = f"client_{client_id}"
    add_pyvis_node(
        net,
        client_node_id,
        f"{client_id} | {client_row['client_name']}",
        "Client",
        title=html_title(
            [
                ("Тип узла", "Client"),
                ("Клиент", client_row["client_name"]),
                ("Сегмент", client_row["segment"]),
                ("Город", client_row["city"]),
                ("Лояльность", client_row["loyalty_score"]),
                ("Риск оттока", client_row["churn_risk"]),
            ]
        ),
        size=28,
    )

    previous_visual_event_id = None
    added_nodes = {client_node_id}

    for event in client_events.to_dict("records"):
        event_visual_id = f"event_{event['event_id']}"
        event_title = html_title(
            [
                ("Тип узла", "Event"),
                ("Обращение", event["event_id"]),
                ("Дата события", event["event_date"].date()),
                ("Этап", event["lifecycle_stage"]),
                ("Тип события", event["event_type"]),
                ("Продукт", event["product"]),
                ("Тема", event["topic"]),
                ("Настроение", event["sentiment"]),
                ("Оценка", event["satisfaction_score"]),
                ("Статус", event["status"]),
            ]
        )
        add_pyvis_node(
            net,
            event_visual_id,
            f"{event['event_order']}. {event['lifecycle_stage']}",
            "Event",
            title=event_title,
            size=22,
        )
        net.add_edge(client_node_id, event_visual_id, label="CLIENT_HAS_EVENT", color="#94a3b8")

        if previous_visual_event_id is not None:
            net.add_edge(previous_visual_event_id, event_visual_id, label="EVENT_NEXT", color="#ef4444", width=3)
        previous_visual_event_id = event_visual_id

        product_node_id = f"product_{event['product']}"
        if product_node_id not in added_nodes:
            add_pyvis_node(
                net,
                product_node_id,
                event["product"],
                "Product",
                title=html_title([("Тип узла", "Product"), ("Название", event["product"])]),
                size=20,
            )
            added_nodes.add(product_node_id)
        net.add_edge(event_visual_id, product_node_id, label="EVENT_ABOUT_PRODUCT", color="#86efac")

        topic_node_id = f"topic_{event['topic']}"
        if topic_node_id not in added_nodes:
            add_pyvis_node(
                net,
                topic_node_id,
                event["topic"],
                "Topic",
                title=html_title([("Тип узла", "Topic"), ("Название", event["topic"])]),
                size=20,
            )
            added_nodes.add(topic_node_id)
        net.add_edge(event_visual_id, topic_node_id, label="EVENT_HAS_TOPIC", color="#f9a8d4")

        stage_node_id = f"stage_{event['lifecycle_stage']}"
        if stage_node_id not in added_nodes:
            add_pyvis_node(
                net,
                stage_node_id,
                event["lifecycle_stage"],
                "LifecycleStage",
                title=html_title([("Тип узла", "LifecycleStage"), ("Название", event["lifecycle_stage"])]),
                size=18,
            )
            added_nodes.add(stage_node_id)
        net.add_edge(event_visual_id, stage_node_id, label="EVENT_HAS_STAGE", color="#c4b5fd")

        event_messages = (
            client_messages.loc[client_messages["event_id"] == event["event_id"]]
            .sort_values("message_order")
            .head(max_messages_per_event)
        )
        for message in event_messages.to_dict("records"):
            message_node_id = f"message_{message['message_id']}"
            add_pyvis_node(
                net,
                message_node_id,
                f"{message['speaker']}: {short_text(message['text'], 45)}",
                "Message",
                title=html_title(
                    [
                        ("Тип узла", "Message"),
                        ("Спикер", message["speaker"]),
                        ("Текст", message["text"]),
                    ]
                ),
                size=12,
            )
            net.add_edge(event_visual_id, message_node_id, label="EVENT_HAS_MESSAGE", color="#cbd5e1")

    return render_pyvis_network(net, output_file, show=show)


client_event_counts = events_df.groupby("client_id").size().sort_values(ascending=False)
sample_client_id = client_event_counts[client_event_counts.between(5, 7)].index[0]

display(Markdown(f"Пример клиента для первичного просмотра: **{sample_client_id}**"))
show_client_lifecycle_graph(sample_client_id, output_file="sample_client_lifecycle_graph.html")

client_options = []
for client in clients_df.to_dict("records"):
    count = int(client_event_counts.loc[client["client_id"]])
    scenario = lifecycle_plan_df.loc[lifecycle_plan_df["client_id"] == client["client_id"], "scenario_name"].iloc[0]
    label = f"{client['client_id']} | {client['client_name']} | {count} событий | {scenario}"
    client_options.append((label, client["client_id"]))

client_dropdown = widgets.Dropdown(options=client_options, description="Клиент:", layout=widgets.Layout(width="95%"))
client_graph_output = widgets.interactive_output(
    lambda selected_client_id: show_client_lifecycle_graph(selected_client_id),
    {"selected_client_id": client_dropdown},
)

display(client_dropdown, client_graph_output)

Пример клиента для первичного просмотра: **C012**

Dropdown(description='Клиент:', layout=Layout(width='95%'), options=(('C001 | Алевтина Игоревна Николаева | 8 …

Output()

## 10. Фрагменты графов разных клиентов

Ниже автоматически выбираются примеры разных жизненных путей: один контакт, короткий путь, обычный путь, длинный путь, жалоба, покупка, повторная покупка и проблемный клиент.

In [10]:
def describe_client_path(client_id):
    client = clients_df.loc[clients_df["client_id"] == client_id].iloc[0]
    client_events = events_df.loc[events_df["client_id"] == client_id].sort_values("event_order")
    scenario_name = client_events["scenario_name"].iloc[0]
    stages = " -> ".join(client_events["lifecycle_stage"].tolist())
    products = ", ".join(sorted(client_events["product"].unique()))
    complaints = int(client_events["is_complaint"].sum())
    purchases = int(client_events["is_purchase"].sum())
    return (
        f"**{client_id} | {client['client_name']}**. "
        f"Сценарий: `{scenario_name}` ({SCENARIO_DESCRIPTIONS[scenario_name]}). "
        f"Количество обращений: {len(client_events)}. "
        f"Продукты: {products}. Жалоб: {complaints}. Покупок: {purchases}. "
        f"Путь: {stages}."
    )


def first_client_by_mask(mask):
    candidates = events_df.loc[mask, "client_id"].drop_duplicates().tolist()
    return candidates[0] if candidates else None


event_counts_by_client = events_df.groupby("client_id").size()
complaint_counts_by_client = events_df.groupby("client_id")["is_complaint"].sum()

fragment_examples = [
    ("Клиент с 1 обращением", event_counts_by_client[event_counts_by_client == 1].index[0], "one_touch"),
    ("Клиент с 2-3 обращениями", event_counts_by_client[event_counts_by_client.between(2, 3)].index[0], "short"),
    ("Клиент с 5-7 обращениями", event_counts_by_client[event_counts_by_client.between(5, 7)].index[0], "medium"),
    ("Клиент с 8+ обращениями", event_counts_by_client[event_counts_by_client >= 8].index[0], "long"),
    ("Клиент с жалобой", first_client_by_mask(events_df["is_complaint"]), "complaint"),
    ("Клиент с покупкой", first_client_by_mask(events_df["event_type"] == "purchase"), "purchase"),
    ("Клиент с повторной покупкой", first_client_by_mask(events_df["event_type"] == "repeat_purchase"), "repeat_purchase"),
    (
        "Проблемный клиент",
        complaint_counts_by_client[complaint_counts_by_client >= 2].sort_values(ascending=False).index[0],
        "problematic",
    ),
]

event_table_columns = [
    "event_order",
    "event_date",
    "scenario_name",
    "lifecycle_stage",
    "event_type",
    "product",
    "topic",
    "sentiment",
    "status",
    "satisfaction_score",
]

for title, client_id, slug in fragment_examples:
    display(Markdown(f"### {title}"))
    display(Markdown(describe_client_path(client_id)))
    display(events_df.loc[events_df["client_id"] == client_id, event_table_columns].sort_values("event_order"))
    show_client_lifecycle_graph(
        client_id,
        output_file=f"fragment_{slug}_{client_id}.html",
        show=True,
        max_messages_per_event=1,
    )

### Клиент с 1 обращением

**C004 | Жукова Прасковья Юльевна**. Сценарий: `one_touch` (клиент обратился один раз и больше не вернулся). Количество обращений: 1. Продукты: Ипотека. Жалоб: 0. Покупок: 0. Путь: знакомство.

,event_order,event_date,scenario_name,lifecycle_stage,event_type,product,topic,sentiment,status,satisfaction_score
16,1,2024-11-21,one_touch,знакомство,introduction,Ипотека,консультация по тарифам,neutral,completed,5


### Клиент с 2-3 обращениями

**C003 | Родионов Амос Аксёнович**. Сценарий: `complaint_and_leave` (жалоба, эскалация и уход после негативного опыта). Количество обращений: 3. Продукты: Кредитная карта, Накопительный счет. Жалоб: 1. Покупок: 0. Путь: использование продукта -> жалоба -> уход клиента.

...

**C005 | Лазарева София Николаевна**. Сценарий: `loyal_multi_product` (лояльный клиент покупает несколько продуктов). Количество обращений: 5. Продукты: Инвестиционный счет, Ипотека, Накопительный счет, Страхование. Жалоб: 0. Покупок: 2. Путь: знакомство -> интерес к продукту -> покупка -> использование продукта -> повторная покупка.

,event_order,event_date,scenario_name,lifecycle_stage,event_type,product,topic,sentiment,status,satisfaction_score
17,1,2024-09-28,loyal_multi_product,знакомство,introduction,Накопительный счет,оформление продукта,neutral,completed,3
18,2,2024-10-16,loyal_multi_product,интерес к продукту,product_interest,Страхование,повышение лимита,neutral,completed,3
19,3,2024-10-28,loyal_multi_product,покупка,purchase,Инвестиционный счет,условия продукта,positive,completed,5
20,4,2024-11-25,loyal_multi_product,использование продукта,usage,Инвестиционный счет,условия продукта,neutral,completed,4
21,5,2024-12-22,loyal_multi_product,повторная покупка,repeat_purchase,Ипотека,оформление продукта,positive,completed,5


### Проблемный клиент

**C015 | Матвеева Оксана Михайловна**. Сценарий: `problematic_many_complaints` (частые жалобы и высокий риск ухода). Количество обращений: 12. Продукты: Дебетовая карта, Накопительный счет. Жалоб: 6. Покупок: 0. Путь: использование продукта -> жалоба -> эскалация -> решение проблемы -> жалоба -> вопрос в поддержку -> жалоба -> эскалация -> закрытие продукта -> уход клиента -> решение проблемы -> жалоба.

,event_order,event_date,scenario_name,lifecycle_stage,event_type,product,topic,sentiment,status,satisfaction_score
73,1,2024-11-05,problematic_many_complaints,использование продукта,usage,Накопительный счет,комиссия,positive,completed,5
74,2,2024-11-15,problematic_many_complaints,жалоба,complaint,Дебетовая карта,просрочка платежа,neutral,escalated,2
75,3,2024-12-18,problematic_many_complaints,эскалация,escalation,Дебетовая карта,техническая ошибка,negative,escalated,2
76,4,2025-01-27,problematic_many_complaints,решение проблемы,problem_resolved,Дебетовая карта,техническая ошибка,positive,resolved,4
77,5,2025-02-16,problematic_many_complaints,жалоба,complaint,Накопительный счет,ошибка в приложении,negative,escalated,1
78,6,2025-03-03,problematic_many_complaints,вопрос в поддержку,support_question,Дебетовая карта,техническая ошибка,positive,resolved,5
79,7,2025-03-30,problematic_many_complaints,жалоба,complaint,Накопительный счет,просрочка платежа,negative,in_progress,2
80,8,2025-04-16,problematic_many_complaints,эскалация,escalation,Накопительный счет,техническая ошибка,neutral,escalated,1
81,9,2025-05-30,problematic_many_complaints,закрытие продукта,product_closure,Дебетовая карта,комиссия,negative,closed,4
82,10,2025-06-16,problematic_many_complaints,уход клиента,churn,Накопительный счет,комиссия,negative,lost,2


## 11. Интерактивный граф вокруг продукта

Функция `show_product_graph(product_name)` показывает продукт, связанные темы, клиентов и обращения. Размер узла темы зависит от количества обращений.

In [11]:
def show_product_graph(product_name, output_file=None, show=True):
    product_events = events_df.loc[events_df["product"] == product_name].copy()
    if output_file is None:
        safe_product = product_name.replace(" ", "_").lower()
        output_file = f"product_graph_{safe_product}.html"

    net = make_network(height="650px")
    product_node_id = f"product_center_{product_name}"

    add_pyvis_node(
        net,
        product_node_id,
        product_name,
        "Product",
        title=html_title(
            [
                ("Тип узла", "Product"),
                ("Название", product_name),
                ("Всего обращений", len(product_events)),
                ("Уникальных клиентов", product_events["client_id"].nunique()),
            ]
        ),
        size=34,
    )

    topic_stats = (
        product_events.groupby("topic")
        .agg(
            event_count=("event_id", "count"),
            unique_clients=("client_id", "nunique"),
            avg_satisfaction=("satisfaction_score", "mean"),
        )
        .reset_index()
        .sort_values("event_count", ascending=False)
    )

    for topic_row in topic_stats.to_dict("records"):
        topic_node_id = f"topic_{topic_row['topic']}"
        topic_size = 16 + min(28, int(topic_row["event_count"]) * 3)
        add_pyvis_node(
            net,
            topic_node_id,
            topic_row["topic"],
            "Topic",
            title=html_title(
                [
                    ("Тип узла", "Topic"),
                    ("Название", topic_row["topic"]),
                    ("Количество обращений", int(topic_row["event_count"])),
                    ("Уникальных клиентов", int(topic_row["unique_clients"])),
                    ("Средняя удовлетворенность", round(topic_row["avg_satisfaction"], 2)),
                ]
            ),
            size=topic_size,
        )
        net.add_edge(product_node_id, topic_node_id, label="PRODUCT_HAS_TOPIC", color="#f9a8d4", width=2)

    for event in product_events.to_dict("records"):
        event_node_id = f"event_{event['event_id']}"
        client_node_id = f"client_{event['client_id']}"
        topic_node_id = f"topic_{event['topic']}"

        add_pyvis_node(
            net,
            event_node_id,
            f"{event['event_order']}. {event['lifecycle_stage']}",
            "Event",
            title=html_title(
                [
                    ("Тип узла", "Event"),
                    ("Дата события", event["event_date"].date()),
                    ("Клиент", event["client_id"]),
                    ("Тема", event["topic"]),
                    ("Настроение", event["sentiment"]),
                    ("Оценка", event["satisfaction_score"]),
                ]
            ),
            size=14,
        )
        add_pyvis_node(
            net,
            client_node_id,
            event["client_id"],
            "Client",
            title=html_title([("Тип узла", "Client"), ("Клиент", event["client_id"])]),
            size=13,
        )

        net.add_edge(product_node_id, event_node_id, label="EVENT_ABOUT_PRODUCT", color="#bbf7d0")
        net.add_edge(event_node_id, topic_node_id, label="EVENT_HAS_TOPIC", color="#f9a8d4")
        net.add_edge(client_node_id, event_node_id, label="CLIENT_HAS_EVENT", color="#bfdbfe")

    return render_pyvis_network(net, output_file, show=show)


default_product = events_df["product"].value_counts().idxmax()
display(Markdown(f"Пример продукта: **{default_product}**"))
show_product_graph(default_product, output_file="product_topic_graph.html")

product_dropdown = widgets.Dropdown(
    options=sorted(events_df["product"].unique().tolist()),
    value=default_product,
    description="Продукт:",
    layout=widgets.Layout(width="70%"),
)
product_graph_output = widgets.interactive_output(
    lambda selected_product: show_product_graph(selected_product),
    {"selected_product": product_dropdown},
)
display(product_dropdown, product_graph_output)

Пример продукта: **Дебетовая карта**

Dropdown(description='Продукт:', layout=Layout(width='70%'), options=('Дебетовая карта', 'Инвестиционный счет'…

Output()

## 12. Интерактивный граф обращений по теме

Функция `show_topic_graph(topic_name)` показывает тему, клиентов, обращения, продукты и сценарии. Рядом выводятся ключевые показатели по выбранной теме.

In [12]:
def topic_metrics_df(topic_events):
    top_products = topic_events["product"].value_counts().head(5)
    top_products_text = "; ".join([f"{product}: {count}" for product, count in top_products.items()])
    negative_share = (topic_events["sentiment"].eq("negative").mean() * 100) if len(topic_events) else 0
    return pd.DataFrame(
        [
            {
                "Всего обращений": len(topic_events),
                "Уникальных клиентов": topic_events["client_id"].nunique(),
                "Частые продукты": top_products_text,
                "Средняя оценка": round(topic_events["satisfaction_score"].mean(), 2) if len(topic_events) else 0,
                "Доля negative, %": round(negative_share, 1),
            }
        ]
    )


def show_topic_graph(topic_name, output_file=None, show=True):
    topic_events = events_df.loc[events_df["topic"] == topic_name].copy()
    if output_file is None:
        safe_topic = topic_name.replace(" ", "_").lower()
        output_file = f"topic_graph_{safe_topic}.html"

    if show:
        display(topic_metrics_df(topic_events))

    net = make_network(height="650px")
    topic_node_id = f"topic_center_{topic_name}"
    add_pyvis_node(
        net,
        topic_node_id,
        topic_name,
        "Topic",
        title=html_title(
            [
                ("Тип узла", "Topic"),
                ("Название", topic_name),
                ("Всего обращений", len(topic_events)),
                ("Уникальных клиентов", topic_events["client_id"].nunique()),
            ]
        ),
        size=34,
    )

    for event in topic_events.to_dict("records"):
        event_node_id = f"event_{event['event_id']}"
        client_node_id = f"client_{event['client_id']}"
        product_node_id = f"product_{event['product']}"
        scenario_node_id = f"scenario_{event['scenario_name']}"

        add_pyvis_node(
            net,
            event_node_id,
            f"{event['event_order']}. {event['lifecycle_stage']}",
            "Event",
            title=html_title(
                [
                    ("Тип узла", "Event"),
                    ("Дата события", event["event_date"].date()),
                    ("Продукт", event["product"]),
                    ("Сценарий", event["scenario_name"]),
                    ("Настроение", event["sentiment"]),
                    ("Оценка", event["satisfaction_score"]),
                ]
            ),
            size=14,
        )
        add_pyvis_node(
            net,
            client_node_id,
            event["client_id"],
            "Client",
            title=html_title([("Тип узла", "Client"), ("Клиент", event["client_id"])]),
            size=13,
        )
        add_pyvis_node(
            net,
            product_node_id,
            event["product"],
            "Product",
            title=html_title([("Тип узла", "Product"), ("Название", event["product"])]),
            size=16,
        )
        add_pyvis_node(
            net,
            scenario_node_id,
            event["scenario_name"],
            "Scenario",
            title=html_title(
                [
                    ("Тип узла", "Scenario"),
                    ("Название", event["scenario_name"]),
                    ("Описание", SCENARIO_DESCRIPTIONS[event["scenario_name"]]),
                ]
            ),
            size=16,
        )

        net.add_edge(client_node_id, event_node_id, label="CLIENT_HAS_EVENT", color="#bfdbfe")
        net.add_edge(event_node_id, topic_node_id, label="EVENT_HAS_TOPIC", color="#f9a8d4")
        net.add_edge(event_node_id, product_node_id, label="EVENT_ABOUT_PRODUCT", color="#bbf7d0")
        net.add_edge(client_node_id, scenario_node_id, label="CLIENT_IN_SCENARIO", color="#a5f3fc")

    return render_pyvis_network(net, output_file, show=show)


default_topic = events_df["topic"].value_counts().idxmax()
display(Markdown(f"Пример темы: **{default_topic}**"))
show_topic_graph(default_topic, output_file="topic_clients_graph.html")

topic_dropdown = widgets.Dropdown(
    options=sorted(events_df["topic"].unique().tolist()),
    value=default_topic,
    description="Тема:",
    layout=widgets.Layout(width="70%"),
)
topic_graph_output = widgets.interactive_output(
    lambda selected_topic: show_topic_graph(selected_topic),
    {"selected_topic": topic_dropdown},
)
display(topic_dropdown, topic_graph_output)

Пример темы: **условия продукта**

,Всего обращений,Уникальных клиентов,Частые продукты,Средняя оценка,"Доля negative, %"
0,46,24,Инвестиционный счет: 11; Потребительский кредит: 11; Страхование: 8; Дебетовая карта: 5; Накопительный счет: 4,4.37,8.7


Dropdown(description='Тема:', index=13, layout=Layout(width='70%'), options=('блокировка карты', 'возврат сред…

Output()

## 13. Общий интерактивный Knowledge Graph

Функция `show_full_knowledge_graph(sample_size=200)` строит общий интерактивный граф. Если граф большой, визуализация ограничивается выборкой узлов.

In [13]:
def select_sample_nodes(sample_size=200):
    if G.number_of_nodes() <= sample_size:
        return list(G.nodes())

    selected = []
    selected_set = set()

    def add_selected(node_id):
        if node_id not in selected_set and len(selected) < sample_size:
            selected.append(node_id)
            selected_set.add(node_id)

    priority_types = ["Product", "Topic", "LifecycleStage", "Scenario", "Segment", "Sentiment"]
    for node_type in priority_types:
        for node_id, attrs in G.nodes(data=True):
            if attrs.get("node_type") == node_type:
                add_selected(node_id)

    client_nodes = [
        node_id
        for node_id, attrs in G.nodes(data=True)
        if attrs.get("node_type") == "Client"
    ]
    client_nodes = sorted(client_nodes, key=lambda node_id: G.degree(node_id), reverse=True)
    for client_node in client_nodes[:25]:
        add_selected(client_node)
        client_events = [
            target
            for _, target, attrs in G.out_edges(client_node, data=True)
            if attrs.get("relation") == "CLIENT_HAS_EVENT"
        ]
        for event_node in client_events[:4]:
            add_selected(event_node)
            for _, neighbor, attrs in G.out_edges(event_node, data=True):
                if attrs.get("relation") in [
                    "EVENT_ABOUT_PRODUCT",
                    "EVENT_HAS_TOPIC",
                    "EVENT_HAS_STAGE",
                    "EVENT_HAS_SENTIMENT",
                ]:
                    add_selected(neighbor)
            message_neighbors = [
                neighbor
                for _, neighbor, attrs in G.out_edges(event_node, data=True)
                if attrs.get("relation") == "EVENT_HAS_MESSAGE"
            ]
            if message_neighbors:
                add_selected(message_neighbors[0])

    return selected


def show_full_knowledge_graph(sample_size=200, output_file="interactive_full_graph.html", show=True):
    sample_nodes = select_sample_nodes(sample_size=sample_size)
    subgraph = G.subgraph(sample_nodes).copy()

    net = make_network(height="750px")
    for node_id, attrs in subgraph.nodes(data=True):
        node_type = attrs.get("node_type", "Unknown")
        title_rows = [
            ("Тип узла", node_type),
            ("Название", attrs.get("label")),
            ("ID", attrs.get("node_id")),
        ]
        for key in ["event_date", "product", "topic", "sentiment", "satisfaction_score", "segment", "city", "scenario_name"]:
            if key in attrs:
                title_rows.append((key, attrs.get(key)))

        add_pyvis_node(
            net,
            node_id,
            attrs.get("label", node_id),
            node_type,
            title=html_title(title_rows),
            size=min(34, 12 + subgraph.degree(node_id) * 2),
        )

    for source, target, attrs in subgraph.edges(data=True):
        net.add_edge(
            source,
            target,
            label=attrs.get("relation", ""),
            color="#cbd5e1",
            width=2 if attrs.get("relation") == "EVENT_NEXT" else 1,
        )

    if show:
        display(Markdown(f"Показано узлов: **{subgraph.number_of_nodes()}** из **{G.number_of_nodes()}**"))
    return render_pyvis_network(net, output_file, show=show, height=750)


show_full_knowledge_graph(sample_size=200, output_file="interactive_full_graph.html")

Показано узлов: **200** из **1808**

PosixPath('interactive_full_graph.html')

## 14. Интерактивная аналитика

Ниже построены интерактивные графики Plotly: распределения, динамика, heatmap, удовлетворенность, жалобы и повторные покупки.

In [14]:
analytics_events_df = events_df.merge(
    clients_df[["client_id", "segment", "city", "churn_risk"]],
    on="client_id",
    how="left",
)

event_counts_df = (
    events_df.groupby("client_id").size().reset_index(name="event_count")
)

fig1 = px.histogram(
    event_counts_df,
    x="event_count",
    nbins=12,
    title="1. Распределение клиентов по количеству обращений",
    labels={"event_count": "Количество обращений", "count": "Количество клиентов"},
)
fig1.update_layout(bargap=0.12)
fig1.show()

product_counts = events_df["product"].value_counts().reset_index()
product_counts.columns = ["product", "event_count"]
fig2 = px.bar(
    product_counts,
    x="product",
    y="event_count",
    title="2. Количество обращений по продуктам",
    labels={"product": "Продукт", "event_count": "Количество обращений"},
)
fig2.update_layout(xaxis_tickangle=-25)
fig2.show()

topic_counts = events_df["topic"].value_counts().reset_index()
topic_counts.columns = ["topic", "event_count"]
fig3 = px.bar(
    topic_counts,
    x="topic",
    y="event_count",
    title="3. Количество обращений по темам",
    labels={"topic": "Тема", "event_count": "Количество обращений"},
)
fig3.update_layout(xaxis_tickangle=-35)
fig3.show()

stage_counts = events_df["lifecycle_stage"].value_counts().reset_index()
stage_counts.columns = ["lifecycle_stage", "event_count"]
fig4 = px.bar(
    stage_counts,
    x="lifecycle_stage",
    y="event_count",
    title="4. Количество обращений по этапам жизненного цикла",
    labels={"lifecycle_stage": "Этап", "event_count": "Количество обращений"},
)
fig4.update_layout(xaxis_tickangle=-35)
fig4.show()

scenario_counts = events_df["scenario_name"].value_counts().reset_index()
scenario_counts.columns = ["scenario_name", "event_count"]
fig5 = px.bar(
    scenario_counts,
    x="scenario_name",
    y="event_count",
    title="5. Количество обращений по сценариям",
    labels={"scenario_name": "Сценарий", "event_count": "Количество обращений"},
)
fig5.update_layout(xaxis_tickangle=-35)
fig5.show()

monthly_events = events_df.groupby("event_month").size().reset_index(name="event_count").sort_values("event_month")
fig6 = px.line(
    monthly_events,
    x="event_month",
    y="event_count",
    markers=True,
    title="6. Динамика обращений по месяцам",
    labels={"event_month": "Месяц", "event_count": "Количество обращений"},
)
fig6.show()

product_topic_matrix = pd.crosstab(events_df["product"], events_df["topic"])
fig7 = px.imshow(
    product_topic_matrix,
    text_auto=True,
    aspect="auto",
    title="7. Heatmap: продукт x тема",
    labels={"x": "Тема", "y": "Продукт", "color": "Количество обращений"},
)
fig7.update_layout(xaxis_tickangle=-35)
fig7.show()

avg_satisfaction_by_product = (
    events_df.groupby("product", as_index=False)["satisfaction_score"].mean().sort_values("satisfaction_score", ascending=False)
)
fig8 = px.bar(
    avg_satisfaction_by_product,
    x="product",
    y="satisfaction_score",
    title="8. Средняя удовлетворенность по продуктам",
    labels={"product": "Продукт", "satisfaction_score": "Средняя оценка"},
    range_y=[0, 5],
)
fig8.update_layout(xaxis_tickangle=-25)
fig8.show()

sentiment_share = events_df["sentiment"].value_counts(normalize=True).mul(100).reset_index()
sentiment_share.columns = ["sentiment", "share"]
fig9 = px.pie(
    sentiment_share,
    names="sentiment",
    values="share",
    title="9. Доля positive / neutral / negative обращений",
)
fig9.show()

fig10 = px.box(
    events_df,
    x="scenario_name",
    y="satisfaction_score",
    points="all",
    title="10. Boxplot удовлетворенности по сценариям",
    labels={"scenario_name": "Сценарий", "satisfaction_score": "Оценка удовлетворенности"},
)
fig10.update_layout(xaxis_tickangle=-35)
fig10.show()

complaints_by_product = (
    events_df.loc[events_df["is_complaint"]]
    .groupby("product")
    .size()
    .reindex(PRODUCTS, fill_value=0)
    .reset_index(name="complaint_count")
)
complaints_by_product.columns = ["product", "complaint_count"]
fig11 = px.bar(
    complaints_by_product,
    x="product",
    y="complaint_count",
    title="11. Количество жалоб по продуктам",
    labels={"product": "Продукт", "complaint_count": "Количество жалоб"},
)
fig11.update_layout(xaxis_tickangle=-25)
fig11.show()

repeat_purchase_by_segment = (
    analytics_events_df.loc[analytics_events_df["event_type"] == "repeat_purchase"]
    .groupby("segment")
    .size()
    .reindex(SEGMENTS, fill_value=0)
    .reset_index(name="repeat_purchase_count")
)
repeat_purchase_by_segment.columns = ["segment", "repeat_purchase_count"]
fig12 = px.bar(
    repeat_purchase_by_segment,
    x="segment",
    y="repeat_purchase_count",
    title="12. Количество повторных покупок по сегментам",
    labels={"segment": "Сегмент", "repeat_purchase_count": "Количество повторных покупок"},
)
fig12.show()

Output hidden; open in https://colab.research.google.com to view.

## 15. Экспорт данных

В конце сохраняются CSV-файлы и основные HTML-файлы с интерактивными графами.

In [15]:
clients_export_df = clients_df.copy()
clients_export_df["base_products"] = clients_export_df["base_products"].apply(lambda values: "; ".join(values))

clients_export_df.to_csv("clients.csv", index=False, encoding="utf-8-sig")
events_df.to_csv("events.csv", index=False, encoding="utf-8-sig")
dialogs_df.to_csv("dialogs.csv", index=False, encoding="utf-8-sig")
nodes_df.to_csv("nodes.csv", index=False, encoding="utf-8-sig")
edges_df.to_csv("edges.csv", index=False, encoding="utf-8-sig")

export_event_counts = events_df.groupby("client_id").size()
export_sample_client_id = export_event_counts[export_event_counts.between(5, 7)].index[0]
export_sample_product = events_df["product"].value_counts().idxmax()
export_sample_topic = events_df["topic"].value_counts().idxmax()

show_full_knowledge_graph(sample_size=200, output_file="interactive_full_graph.html", show=False)
show_client_lifecycle_graph(export_sample_client_id, output_file="sample_client_lifecycle_graph.html", show=False)
show_product_graph(export_sample_product, output_file="product_topic_graph.html", show=False)
show_topic_graph(export_sample_topic, output_file="topic_clients_graph.html", show=False)

exported_files = [
    "clients.csv",
    "events.csv",
    "dialogs.csv",
    "nodes.csv",
    "edges.csv",
    "interactive_full_graph.html",
    "sample_client_lifecycle_graph.html",
    "product_topic_graph.html",
    "topic_clients_graph.html",
]

exported_summary = pd.DataFrame(
    {
        "file": exported_files,
        "exists": [Path(file_name).exists() for file_name in exported_files],
        "size_bytes": [Path(file_name).stat().st_size if Path(file_name).exists() else 0 for file_name in exported_files],
    }
)
display(exported_summary)

,file,exists,size_bytes
0,clients.csv,True,9193
1,events.csv,True,58181
2,dialogs.csv,True,441340
3,nodes.csv,True,595625
4,edges.csv,True,235117
5,interactive_full_graph.html,True,965033
6,sample_client_lifecycle_graph.html,True,728292
7,product_topic_graph.html,True,782247
8,topic_clients_graph.html,True,789233


## 16. Автоматические проверки качества

Блок `assert` проверяет размер данных, разнообразие клиентских путей, связность событий, наличие сообщений и базовые элементы Knowledge Graph.

In [16]:
assert len(clients_df) == 50, "clients_df должен содержать 50 клиентов"
assert not events_df.empty, "events_df не должен быть пустым"
assert not dialogs_df.empty, "dialogs_df не должен быть пустым"

quality_event_counts = events_df.groupby("client_id").size()
assert quality_event_counts.nunique() > 1, "У клиентов должно быть разное количество обращений"
assert (quality_event_counts == 1).sum() >= 8, "Должно быть минимум 8 клиентов с 1 обращением"
assert quality_event_counts.between(2, 3).sum() >= 10, "Должно быть минимум 10 клиентов с 2-3 обращениями"
assert quality_event_counts.between(5, 7).sum() >= 15, "Должно быть минимум 15 клиентов с 5-7 обращениями"
assert (quality_event_counts >= 8).sum() >= 5, "Должно быть минимум 5 клиентов с 8+ обращениями"

path_signatures = events_df.groupby("client_id")["event_type"].apply(lambda values: tuple(values.tolist()))
assert path_signatures.nunique() > 1, "Не у всех клиентов должен быть одинаковый жизненный путь"

message_counts_by_event = dialogs_df.groupby("event_id").size()
assert set(events_df["event_id"]).issubset(set(message_counts_by_event.index)), "У каждого события должно быть хотя бы одно сообщение"
assert (message_counts_by_event >= 1).all(), "У каждого события должно быть хотя бы одно сообщение"

for client_id, group in events_df.sort_values(["client_id", "event_order"]).groupby("client_id"):
    ordered_group = group.sort_values("event_order").reset_index(drop=True)
    assert ordered_group["event_date"].is_monotonic_increasing, f"Даты событий клиента {client_id} должны идти по порядку"
    previous_values = ordered_group["previous_event_id"].tolist()
    event_ids = ordered_group["event_id"].tolist()
    assert previous_values[0] is None or pd.isna(previous_values[0]), f"Первое событие клиента {client_id} должно иметь previous_event_id = None"
    for idx in range(1, len(ordered_group)):
        assert previous_values[idx] == event_ids[idx - 1], f"Некорректный previous_event_id у клиента {client_id}"

graph_node_types = set(nodes_df["node_type"])
for required_node_type in ["Client", "Event", "Message", "Product", "Topic"]:
    assert required_node_type in graph_node_types, f"В графе должен быть тип узла {required_node_type}"

graph_relations = set(edges_df["relation"])
for required_relation in ["CLIENT_HAS_EVENT", "EVENT_NEXT", "EVENT_ABOUT_PRODUCT", "EVENT_HAS_TOPIC"]:
    assert required_relation in graph_relations, f"В графе должна быть связь {required_relation}"

key_fields = ["client_id", "event_id", "event_date", "product", "topic"]
assert not events_df[key_fields].isna().any().any(), "В ключевых полях событий не должно быть пропусков"

print("Все проверки успешно пройдены")

Все проверки успешно пройдены


## 17. Вывод

В работе создан генератор синтетических жизненных циклов клиентов.

В отличие от первой версии, у клиентов теперь разные сценарии поведения: часть клиентов обращается один раз, часть проходит обычный путь из 5-7 обращений, а часть имеет длинную цепочку на несколько месяцев. Отдельно представлены клиенты с жалобами, повторными покупками, решением проблемы, удержанием и уходом после негативного опыта.

По каждому обращению сгенерированы синтетические русскоязычные диалоги. Построен граф знаний, связывающий клиентов, обращения, продукты, темы, этапы жизненного цикла, сценарии и сообщения.

Интерактивные графы позволяют смотреть жизненный путь одного клиента, анализировать продукт и связанные с ним темы, а также изучать обращения клиентов по конкретной теме. Аналитические графики помогают оценивать жалобы, покупки, повторные обращения, удовлетворенность и динамику обращений по месяцам.

Такой набор данных можно использовать для тестирования Knowledge Graph, GraphRAG и клиентской аналитики.